# ViHSD Project

This notebook **is** the project: every step of the Vietnamese hate-speech pipeline — data
loading, fold geometry, classical baselines, transformer fine-tuning, metrics, artifacts, and a
final model-vs-model comparison — written out inline, cell by cell.

| Stage | What happens here | Models |
|---|---|---|
| 1 · Input | load `uitnlp/vihsd`, profile classes and lengths, audit the fold plan | — |
| 2 · Classical | TF-IDF word+char n-grams → balanced linear model, 5-fold CV | `logreg`, `svm` |
| 3 · Transformer | Hugging Face `Trainer`, weighted cross-entropy, 5-fold CV | `uitnlp/visobert`, `xlm-roberta-base`, `vinai/phobert-base` |
| 4 · Output | per-fold metrics, confusion matrices | all |
| 5 · Comparison | leaderboard, paired per-fold significance tests, verdict vs published ViHSD numbers | all |

> **Content note.** ViHSD is a hate-speech benchmark. Stage 1 quotes a few comments verbatim,
> including hostile ones, because the pipeline is required to leave raw text untouched — no
> stripping of diacritics, emoji, slang, or punctuation.


| | |
|---|---|
| **Requires** | Python ≥ 3.12 · network access to Hugging Face (`uitnlp/vihsd` + model weights) |
| **Stages 1–2 (classical)** | CPU; ~2–3 min for the full 24,048-example split |
| **Stage 3 (transformer)** | GPU; ~3–8 min per fold per model — on a CPU-only machine this stage prints a notice and skips itself |

Set `SAMPLE_SIZE = 2000` in the configuration cell for a fast smoke run.


In [16]:
!pip install -q \
  "transformers==4.57.6" \
  "huggingface-hub==0.36.2" \
  "diffusers==0.35.2" \
  "gradio==5.50.0"

KeyboardInterrupt: 

In [ ]:
# Environment check — surface a missing dependency here, not three stages deep.
import sys
from pathlib import Path
import sklearn
import torch
import transformers

print(f"python       : {sys.version.split()[0]}")
print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"scikit-learn : {sklearn.__version__}")
if torch.cuda.is_available():
    print(f"device       : cuda — {torch.cuda.get_device_name(0)}")
else:
    print("device       : cpu — fine for stages 1/2; stage 3 will skip itself")


## Configuration

Every knob in one place. The two invariants that keep results comparable are **`FOLDS = 5`** and
**`SEED = 42`**: all four models share exactly the same folds, which is what makes the paired
per-fold comparison in stage 5 valid. Changing them invalidates the comparison against the
project's accepted baseline.


In [ ]:
from pathlib import Path
import os

# --- CV geometry: fixed project-wide, keep as-is for comparability ---
SPLIT = "train"      # "train" (24,048) | "validation" (2,672) | "test" (6,680)
FOLDS = 5
SEED = 42
SAMPLE_SIZE = None   # first N examples, e.g. 2000 for a smoke run; None = the whole split

# --- Stage 2: classical TF-IDF baselines (CPU, ~70 s per model on the full split) ---
CLASSICAL_MODELS = ("logreg", "svm")   # svm exposes no probabilities -> no ROC/PR-AUC

# --- Stage 3: transformer fine-tuning (GPU). These are the three backbones the project benchmarks
TRANSFORMER_MODELS = (
    {"model_name": "vinai/phobert-base", "freeze_embeddings": False},
    {"model_name": "uitnlp/visobert", "freeze_embeddings": False},
    {"model_name": "xlm-roberta-base", "freeze_embeddings": False},
)
EPOCHS = 3.0
GRAD_ACCUM_STEPS = 1
LEARNING_RATE = 2e-5
MAX_LENGTH = 96
WARMUP_RATIO = 0.1
OPTIM = "adamw_torch"

if os.path.exists("/kaggle/working"):
    OUT_ROOT = Path("/kaggle/working/outputs")   # Kaggle
else:
    OUT_ROOT = Path("/content/outputs")          # Colab

## Stage 1 — Input: the ViHSD dataset

**Input:** Hugging Face `uitnlp/vihsd`, ~33.4K Vietnamese social-media comments labelled
`CLEAN` (0) / `OFFENSIVE` (1) / `HATE` (2). The loader infers the text and label columns
(`free_text`, `label_id`) and normalizes **only the label**; the text reaches every model exactly
as written.

The ~12 : 1.6 : 1 class imbalance is why every model below is class-balanced, and why
**Macro F1 — not accuracy — is the primary metric**: accuracy is inflatable by the 83% majority
class.


In [ ]:
from datasets import load_dataset
from google.colab import userdata
from huggingface_hub import login

LABEL_NAMES = ("CLEAN", "OFFENSIVE", "HATE")

hf_token = userdata.get('HF_Token')
login(token=hf_token)


def normalize_label(raw_label):
    # Label parsing is the ONLY normalization this project allows.
    if isinstance(raw_label, int) and not isinstance(raw_label, bool):
        if raw_label in (0, 1, 2):
            return raw_label
        raise ValueError(f"unknown ViHSD label: {raw_label}")
    if isinstance(raw_label, str):
        key = raw_label.strip().upper()
        if key in LABEL_NAMES:
            return LABEL_NAMES.index(key)
        if key.isdigit():
            return normalize_label(int(key))
        raise ValueError(f"unknown ViHSD label: {raw_label}")
    raise ValueError(f"unknown ViHSD label: {raw_label}")


dataset = load_dataset("uitnlp/vihsd", split=SPLIT)
text_column = next(
    (c for c in ("text", "comment", "content", "sentence", "free_text") if c in dataset.column_names),
    None,
)
label_column = next(
    (c for c in ("label", "labels", "class", "category", "label_id") if c in dataset.column_names),
    None,
)
if text_column is None or label_column is None:
    raise RuntimeError(f"could not infer text/label columns from: {dataset.column_names}")

rows = dataset if SAMPLE_SIZE is None else dataset.select(range(min(SAMPLE_SIZE, len(dataset))))
texts = [str(row[text_column]) for row in rows]
labels = [normalize_label(row[label_column]) for row in rows]
print(f"uitnlp/vihsd split={SPLIT!r} columns=({text_column}, {label_column}) -> {len(texts):,} examples")


In [ ]:
%matplotlib inline
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

MARK = "#1d4ed8"
counts = Counter(labels)
total = len(labels)

distribution = pd.DataFrame(
    {
        "label_id": list(range(len(LABEL_NAMES))),
        "label": list(LABEL_NAMES),
        "count": [counts.get(label_id, 0) for label_id in range(len(LABEL_NAMES))],
    }
)
distribution["share"] = [f"{count / total:.2%}" for count in distribution["count"]]
word_lengths = pd.Series([len(text.split()) for text in texts])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].bar(distribution["label"], distribution["count"], color=MARK, width=0.62)
axes[0].set_title(
    f"CLEAN dominates the {SPLIT} split: {distribution['share'].iloc[0]} of {total:,} comments",
    fontsize=10,
)
axes[0].set_ylabel("comments")
for label, count in zip(distribution["label"], distribution["count"], strict=True):
    axes[0].text(label, count, f"{count:,}", ha="center", va="bottom", fontsize=9)

axes[1].hist(word_lengths.clip(upper=60), bins=60, color=MARK)
axes[1].set_title(
    f"Comments are short: median {int(word_lengths.median())} words, "
    f"p99 {int(word_lengths.quantile(0.99))} — why max_length=160 is enough",
    fontsize=10,
)
axes[1].set_xlabel("words per comment (clipped at 60)")

for ax in axes:
    ax.grid(axis="y", color="#e2e8f0", linewidth=0.8)
    ax.set_axisbelow(True)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
fig.tight_layout()
plt.show()
distribution


In [ ]:
# Verbatim samples, two per class: diacritics, emoji, teencode and repeated punctuation reach
# the models exactly as written, because raw text is never modified.
sample_rows = []
for label_id, label in enumerate(LABEL_NAMES):
    for index in [i for i, lab in enumerate(labels) if lab == label_id][:2]:
        sample_rows.append({"label": f"{label_id} · {label}", "text": texts[index][:140]})
display(pd.DataFrame(sample_rows))


In [ ]:
from sklearn.model_selection import StratifiedKFold

# One splitter, shared by every model below: fold i of one model holds out exactly the same
# comments as fold i of any other, which is what stage 5's paired tests rely on.
splitter = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
fold_plan = []
for fold, (train_idx, test_idx) in enumerate(splitter.split(texts, labels), start=1):
    held_out = Counter(labels[idx] for idx in test_idx)
    fold_plan.append(
        {
            "fold": fold,
            "train": len(train_idx),
            "held out": len(test_idx),
            **{
                f"held-out {label.lower()}": held_out.get(label_id, 0)
                for label_id, label in enumerate(LABEL_NAMES)
            },
        }
    )
print(f"StratifiedKFold(n_splits={FOLDS}, shuffle=True, random_state={SEED})")
pd.DataFrame(fold_plan)


In [ ]:
import json
from statistics import fmean, pstdev

import numpy as np
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_recall_fscore_support,
    roc_auc_score,
    precision_recall_curve,
    roc_curve,
)
from sklearn.preprocessing import label_binarize


def evaluate_fold(fold, y_true, y_pred, y_prob, out_dir):
    class_ids = [0, 1, 2]
    precision, recall, f1, _support = precision_recall_fscore_support(
        y_true, y_pred, labels=class_ids, average=None, zero_division=0
    )
    scalars = {
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    }
    for idx, name in enumerate(LABEL_NAMES):
        scalars[f"{name.lower()}_precision"] = float(precision[idx])
        scalars[f"{name.lower()}_recall"] = float(recall[idx])
        scalars[f"{name.lower()}_f1"] = float(f1[idx])
    if y_prob is not None:
        truth = label_binarize(y_true, classes=class_ids)
        probabilities = np.asarray(y_prob, dtype=float)
        scalars["roc_auc_ovr_weighted"] = float(
            roc_auc_score(truth, probabilities, average="weighted", multi_class="ovr")
        )
        scalars["pr_auc_macro"] = float(average_precision_score(truth, probabilities, average="macro"))
    matrix = confusion_matrix(y_true, y_pred, labels=class_ids).astype(int).tolist()
    return {"fold": fold, "scalars": scalars, "confusion_matrix": matrix}


def summarize_folds(fold_scalars):
    names = sorted({name for scalars in fold_scalars for name in scalars})
    return {
        name: {
            "mean": fmean([s[name] for s in fold_scalars if name in s]),
            "std": pstdev([s[name] for s in fold_scalars if name in s]),
        }
        for name in names
    }


## Stage 2 — Classical baselines: TF-IDF + linear model

Per fold, everything is fit **inside the training fold only** (leak-free by construction):

```
TF-IDF words (1–2 grams, min_df=2)  ∪  TF-IDF chars (3–5 grams, min_df=2)
        └──>  LogisticRegression(class_weight="balanced")   or   LinearSVC(balanced)
```

The char n-grams are what catch obfuscated profanity and teencode; `class_weight="balanced"`
is the counterweight to the 12 : 1.6 : 1 imbalance.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC


def build_classical_pipeline(model, seed):
    features = FeatureUnion(
        [
            ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2)),
            ("char", TfidfVectorizer(analyzer="char", ngram_range=(3, 5), min_df=2)),
        ]
    )
    if model == "logreg":
        classifier = LogisticRegression(class_weight="balanced", max_iter=1_000, random_state=seed)
    else:  # "svm"
        classifier = LinearSVC(class_weight="balanced", random_state=seed)
    return Pipeline([("features", features), ("classifier", classifier)])


def run_classical_cv(model, out_dir):
    fold_results = []
    for fold, (train_idx, test_idx) in enumerate(splitter.split(texts, labels), start=1):
        pipeline = build_classical_pipeline(model, SEED)
        pipeline.fit([texts[i] for i in train_idx], [labels[i] for i in train_idx])
        test_texts = [texts[i] for i in test_idx]
        test_labels = [labels[i] for i in test_idx]
        predictions = pipeline.predict(test_texts)
        probabilities = pipeline.predict_proba(test_texts) if hasattr(pipeline, "predict_proba") else None
        fold_results.append(evaluate_fold(fold, test_labels, predictions, probabilities, out_dir))
    return fold_results


In [ ]:
import time

HEADLINE_METRICS = (
    "macro_f1",
    "weighted_f1",
    "balanced_accuracy",
    "mcc",
    "roc_auc_ovr_weighted",
    "pr_auc_macro",
)

all_runs = {}

for model in CLASSICAL_MODELS:
    out_dir = OUT_ROOT / f"classical_{model}"
    started = time.perf_counter()
    all_runs[f"classical_{model}"] = run_classical_cv(model, out_dir)
    macro = [r["scalars"]["macro_f1"] for r in all_runs[f"classical_{model}"]]

def fold_table(fold_results):
    # One row per fold; a metric a model cannot produce is marked, never approximated.
    rows = []
    for result in fold_results:
        row = {"fold": result["fold"]}
        for metric in HEADLINE_METRICS:
            row[metric] = round(result["scalars"][metric], 4) if metric in result["scalars"] else "n/a"
        rows.append(row)
    return pd.DataFrame(rows)


for run_name, fold_results in all_runs.items():
    print(run_name)
    display(fold_table(fold_results))


## Stage 3 — Transformer fine-tuning (needs a GPU)

Same folds, same seed, same metrics — only the backbone changes. Per fold:

1. tokenize with `max_length=160` and **dynamic per-batch padding** (the median comment is
   ~8 words, so padding every sample to 160 would waste most of each batch);
2. `AutoModelForSequenceClassification(num_labels=3)`;
3. Hugging Face `Trainer` with AdamW, a linear warmup schedule (`warmup_ratio=0.1`), `fp16` on
   CUDA, and a cross-entropy loss **weighted by this fold's training-label frequencies only**;




In [ ]:
import torch
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)


def load_tokenizer(model_name):
    # Slow tokenizer first: the fast ones for PhoBERT/XLM-R are why the project pins
    # transformers<5.
    try:
        return AutoTokenizer.from_pretrained(model_name, use_fast=False)
    except (ValueError, OSError, ImportError):
        return AutoTokenizer.from_pretrained(model_name, use_fast=True)


def class_weights(train_labels):
    # Inverse class frequency from THIS fold's training labels only — leak-free.
    counts = torch.bincount(torch.tensor(train_labels), minlength=3).float()
    return counts.sum() / (counts.clamp_min(1.0) * 3.0)


def weighted_loss_trainer_class(weights):
    class WeightedLossTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            batch_labels = inputs.pop("labels")
            outputs = model(**inputs)
            loss_fn = torch.nn.CrossEntropyLoss(weight=weights.to(outputs.logits.device))
            loss = loss_fn(outputs.logits.view(-1, 3), batch_labels.view(-1))
            return (loss, outputs) if return_outputs else loss

    return WeightedLossTrainer


def macro_f1_metric(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=-1)
    return {
        "macro_f1": float(f1_score(eval_pred.label_ids, predictions, average="macro", zero_division=0))
    }


def epoch_history(trainer):
    by_epoch = {}
    for record in trainer.state.log_history:
        epoch = record.get("epoch")
        if epoch is None:
            continue
        row = by_epoch.setdefault(round(float(epoch), 4), {"epoch": round(float(epoch), 4)})
        for key in ("loss", "eval_loss", "eval_macro_f1"):
            if key in record:
                row[key] = float(record[key])
    return [by_epoch[key] for key in sorted(by_epoch)]


In [ ]:
import time

BATCH_SIZE = 512 # GPU A100 - 40GB VRAM consider adjust down for lower GPU model

def run_transformer_cv(run_config, out_dir):
    model_name = run_config["model_name"]
    tokenizer = load_tokenizer(model_name)
    collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)
    use_fp16 = torch.cuda.is_available()
    fold_results, history = [], []

    for fold, (train_idx, test_idx) in enumerate(splitter.split(texts, labels), start=1):
        train_labels = [labels[i] for i in train_idx]
        test_labels = [labels[i] for i in test_idx]

        def tokenize(batch):
            return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

        train_ds = Dataset.from_dict({"text": [texts[i] for i in train_idx], "label": train_labels})
        test_ds = Dataset.from_dict({"text": [texts[i] for i in test_idx], "label": test_labels})
        train_ds = train_ds.map(tokenize, batched=True, remove_columns=["text"])
        test_ds = test_ds.map(tokenize, batched=True, remove_columns=["text"])

        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)
        if run_config["freeze_embeddings"]:
            for parameter in model.get_input_embeddings().parameters():
                parameter.requires_grad = False
        args = TrainingArguments(
            output_dir=str(out_dir / f"fold_{fold}"),
            eval_strategy="epoch",
            logging_strategy="epoch",
            save_strategy="no",
            learning_rate=LEARNING_RATE,
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUM_STEPS,
            num_train_epochs=EPOCHS,
            lr_scheduler_type="linear",
            warmup_ratio=WARMUP_RATIO,
            optim=OPTIM,
            gradient_checkpointing=False,
            fp16=use_fp16,
            seed=SEED,
            report_to=[],
        )
        trainer = weighted_loss_trainer_class(class_weights(train_labels))(
            model=model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=test_ds,
            data_collator=collator,
            compute_metrics=macro_f1_metric,
        )
        trainer.train()
        fold_history = epoch_history(trainer)
        history.append({"fold": fold, "epochs": fold_history})

        output = trainer.predict(test_ds)
        probabilities = torch.softmax(torch.tensor(output.predictions), dim=1).numpy()
        predictions = np.argmax(probabilities, axis=1)
        result = evaluate_fold(fold, test_labels, predictions.tolist(), probabilities.tolist(), out_dir)
        fold_results.append(result)
        print(f"  fold {fold}/{FOLDS}: macro_f1 {result['scalars']['macro_f1']:.4f}", flush=True)
        # Save model
        trainer.save_model(out_dir)
        tokenizer.save_pretrained(out_dir)

        del model, trainer
        if use_fp16:
            torch.cuda.empty_cache()
    return fold_results

if not torch.cuda.is_available():
    print("No CUDA device: stage 3 needs a GPU, so it skips itself here.")
    print("Re-run on a GPU machine for the transformer half of the comparison.")
else:
    for run_config in TRANSFORMER_MODELS:
        slug = run_config["model_name"].rsplit("/", 1)[-1]
        out_dir = OUT_ROOT / f"transformer_{slug}"
        started = time.perf_counter()
        print(
            f"{run_config['model_name']} (freeze_embeddings={run_config['freeze_embeddings']})",
            flush=True,
        )
        all_runs[f"transformer_{slug}"] = run_transformer_cv(run_config, out_dir)
        macro = [r["scalars"]["macro_f1"] for r in all_runs[f"transformer_{slug}"]]
        print(
            f"  done in {time.perf_counter() - started:6.1f}s   macro_f1 per fold: "
            f"{[round(v, 4) for v in macro]}",
            flush=True,
        )


## Stage 4 — Output: summarize



In [ ]:
summaries = {}
for run_name, fold_results in all_runs.items():
    summary = summarize_folds([result["scalars"] for result in fold_results])
    summaries[run_name] = summary
    lines = ["# Evaluation Summary", "", "| Metric | Mean | Std |", "|---|---:|---:|"]
    lines += [f"| {name} | {s['mean']:.4f} | {s['std']:.4f} |" for name, s in sorted(summary.items())]
summary_table = pd.DataFrame(
    {
        run_name: {
            metric: f"{stats['mean']:.4f} ± {stats['std']:.4f}"
            for metric in HEADLINE_METRICS
            if (stats := summary.get(metric)) is not None
        }
        for run_name, summary in summaries.items()
    }
)
summary_table.index.name = "mean ± std over folds"
summary_table.fillna("n/a")


In [ ]:
from matplotlib.colors import LinearSegmentedColormap

# Sequential ramp for an ordered magnitude; every cell also prints its value, so nothing is
# encoded by colour alone.
RAMP = LinearSegmentedColormap.from_list("vihsd_blues", ["#60a5fa", "#2563eb", "#1e3a8a"])


def mean_confusion(fold_results):
    return np.array([r["confusion_matrix"] for r in fold_results], dtype=float).mean(axis=0)


if not all_runs:
    print("No stage ran — nothing to plot.")
else:
    fig, axes = plt.subplots(1, len(all_runs), figsize=(4.3 * len(all_runs), 3.9), squeeze=False)
    for ax, (run_name, fold_results) in zip(axes[0], all_runs.items(), strict=True):
        matrix = mean_confusion(fold_results)
        share = matrix / matrix.sum(axis=1, keepdims=True)  # each row = one true class
        image = ax.imshow(share, cmap=RAMP, vmin=0.0, vmax=1.0)
        ax.set_xticks(range(len(LABEL_NAMES)), LABEL_NAMES, fontsize=8)
        ax.set_yticks(range(len(LABEL_NAMES)), LABEL_NAMES, fontsize=8)
        ax.set_xlabel("predicted", fontsize=9)
        ax.set_title(run_name, fontsize=10)
        for row in range(len(LABEL_NAMES)):
            for col in range(len(LABEL_NAMES)):
                ax.text(
                    col,
                    row,
                    f"{share[row, col]:.2f}",
                    ha="center",
                    va="center",
                    fontsize=9,
                    color="white" if share[row, col] > 0.55 else "#0f172a",
                )
        fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    fig.suptitle(
        "CLEAN is near-solved; OFFENSIVE is the bottleneck — mean over folds, rows normalized "
        "by true class",
        fontsize=11,
    )
    fig.tight_layout()
    plt.show()


## Stage 5 — Comparison: which model wins, and by how much?

Training and testing are complete for every model above, so now they are compared on equal
terms. Three views:

1. **Leaderboard** — every headline metric, mean ± std over the 5 folds, ranked by Macro F1;
2. **Paired per-fold significance** — because all models saw the same seed-13 folds, fold *i* of
   one model pairs with fold *i* of another; a paired t-test (and the Wilcoxon signed-rank test)
   over those 5 pairs is the project's decision rule for calling a winner;
3. **Verdict vs published ViHSD numbers** — the gates the project set for the transformer phase.


In [ ]:
leaderboard = pd.DataFrame(
    {
        run_name: {metric: summary[metric]["mean"] for metric in HEADLINE_METRICS if metric in summary}
        for run_name, summary in summaries.items()
    }
).T.sort_values("macro_f1", ascending=False)
leaderboard.insert(0, "rank", range(1, len(leaderboard) + 1))
print("Leaderboard — Macro F1 decides the ranking; the rest is diagnostic.")
leaderboard.round(4).fillna("n/a")   # SVM has no probabilities, hence no AUC metrics


In [ ]:
from scipy import stats

REFERENCE_MODEL = "classical_logreg"   # the project's accepted baseline (HYPE-4)
reference = [r["scalars"]["macro_f1"] for r in all_runs[REFERENCE_MODEL]]

paired_rows = []
for run_name, fold_results in all_runs.items():
    if run_name == REFERENCE_MODEL:
        continue
    observed = [r["scalars"]["macro_f1"] for r in fold_results]
    deltas = [b - a for a, b in zip(reference, observed, strict=True)]
    paired_rows.append(
        {
            "model": run_name,
            "mean paired Δ macro_f1": round(fmean(deltas), 4),
            "folds better": sum(d > 0 for d in deltas),
            "paired t p": round(float(stats.ttest_rel(observed, reference).pvalue), 4),
            "wilcoxon p": round(float(stats.wilcoxon(observed, reference).pvalue), 4),
        }
    )
print(f"Paired against {REFERENCE_MODEL} over the {FOLDS} shared folds:")
display(pd.DataFrame(paired_rows).set_index("model"))
print("With n=5 folds the Wilcoxon signed-rank test cannot reach p<0.05 (its floor is")
print("0.0625); the paired t-test can. Treat 'folds better' as the robust signal.")


In [ ]:
MODEL_COLORS = {
    "classical_logreg": "#1d4ed8",
    "classical_svm": "#b45309",
    "transformer_visobert": "#166534",
    "transformer_xlm-roberta-base": "#7c3aed",
}
names = list(leaderboard.index)
means = [summaries[name]["macro_f1"]["mean"] for name in names]

fig, ax = plt.subplots(figsize=(9, 4.2))
xs = list(range(len(names)))
bars = ax.bar(xs, means, color=[MODEL_COLORS.get(name, "#1d4ed8") for name in names], width=0.6)
for x, name in zip(xs, names, strict=True):
    ax.scatter(
        [x] * FOLDS,
        [r["scalars"]["macro_f1"] for r in all_runs[name]],
        color="#0f172a",
        s=14,
        zorder=3,
        alpha=0.75,
    )
for bar, mean in zip(bars, means, strict=True):
    ax.text(bar.get_x() + bar.get_width() / 2, mean, f"{mean:.4f}", ha="center", va="bottom", fontsize=9)
ax.axhline(0.6475, color="#64748b", linestyle="--", linewidth=1)
ax.text(
    0.99,
    0.97,
    "dashed line = accepted classical baseline 0.6475",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=8,
    color="#475569",
)
ax.set_xticks(xs, [name.replace("transformer_", "tf: ").replace("classical_", "clf: ") for name in names], fontsize=9)
ax.set_ylabel("Macro F1")
ax.set_ylim(0, max(means) * 1.18)
ax.set_title("Macro F1 by model — bars are the 5-fold mean, dots the individual folds", fontsize=11)
ax.grid(axis="y", color="#e2e8f0", linewidth=0.8)
ax.set_axisbelow(True)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
fig.tight_layout()
plt.show()


In [ ]:
PUBLISHED_MACRO_F1 = {
    "classical_logreg": "0.52-0.56 (classical TF-IDF)",
    "classical_svm": "0.52-0.56 (classical TF-IDF)",
    "transformer_visobert": "0.6771 (ViSoBERT, SOTA)",
    "transformer_xlm-roberta-base": "0.6550 (XLM-R base)",
}
BEATS_CLASSICAL_GATE = 0.67   # recalibrated HYPE-4 gate: "beats classical"
SOTA_GATE = 0.68              # stretch goal: "SOTA-level"

verdict_rows = []
for run_name in leaderboard.index:
    mean = summaries[run_name]["macro_f1"]["mean"]
    verdict_rows.append(
        {
            "model": run_name,
            "macro_f1": round(mean, 4),
            "published reference": PUBLISHED_MACRO_F1.get(run_name, "n/a"),
            "beats classical (>=0.67)": "yes" if mean >= BEATS_CLASSICAL_GATE else "no",
            "SOTA-level (>=0.68)": "yes" if mean >= SOTA_GATE else "no",
        }
    )
display(pd.DataFrame(verdict_rows).set_index("model"))

best = leaderboard.index[0]
print(f"Best model on this run: {best} (Macro F1 {summaries[best]['macro_f1']['mean']:.4f}).")


## Stage 6 — Deployment best model

In [ ]:
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

SAVE_DIR = Path(f"/content/drive/MyDrive/{OUT_ROOT}/transformer_visobert")

In [ ]:
import shutil
from pathlib import Path

if SAVE_DIR.exists():
    shutil.rmtree(SAVE_DIR)
shutil.copytree(Path(OUT_ROOT/"transformer_visobert"), SAVE_DIR)

print("Saved to:", SAVE_DIR)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = SAVE_DIR
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

LABELS = {
    0: "CLEAN",
    1: "OFFENSIVE",
    2: "HATE",
}

def predict_detailed(text):
    text = text.strip()

    if not text:
        return "Please enter some text.", {}

    inputs = tokenizer(
        text,
        truncation=True,
        max_length=160,
        return_tensors="pt"
    ).to(device)

    with torch.inference_mode():
        logits = model(**inputs).logits

    probs = torch.softmax(logits, dim=-1)[0].cpu()

    scores = {
        LABELS[i]: float(probs[i])
        for i in range(len(probs))
    }

    pred_id = int(torch.argmax(probs))
    prediction = LABELS[pred_id]

    return prediction, scores

In [ ]:
import gradio as gr

with gr.Blocks() as demo:
    gr.Markdown("# Vietnamese Hate Speech Detection")

    text_input = gr.Textbox(
        label="Vietnamese text",
        placeholder="Nhập nội dung cần kiểm tra...",
        lines=5
    )

    predict_button = gr.Button("Analyze")

    predicted_class = gr.Textbox(
        label="Predicted class"
    )

    probabilities = gr.Label(
        label="Class probabilities",
        num_top_classes=3
    )

    predict_button.click(
        fn=predict_detailed,
        inputs=text_input,
        outputs=[
            predicted_class,
            probabilities
        ]
    )

demo.launch()